# NJ inbound weights: other NJ ZCTAs and New York City

Workplace is always a New Jersey ZCTA. Home is another New Jersey ZCTA, or a New York City ZCTA.

**Files to keep in `Dhruvesh/Data/LODES`**
- `nj_od_main_JT00_YEAR.csv.gz` — home and work both in New Jersey (census blocks)
- `nj_od_aux_JT00_YEAR.csv.gz` — work in New Jersey, home in another state (blocks)
- `nj_xwalk.csv.gz` and `ny_xwalk.csv.gz` — block to ZCTA

`S000` is jobs on that block pair. The steps below add those jobs up to ZCTA pairs.

To switch year, change `YEAR` in `build_nj_inbound_weights.py` and put that year's two OD files in `Data/LODES`. Crosswalks stay as they are.

Run this notebook from the project root.


## 1. Load the block-to-ZCTA map

The OD files do not contain ZIP codes. The crosswalk is what turns a census block into a ZCTA.


In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, "Dhruvesh")
from build_nj_inbound_weights import YEAR, block_lookup, build_flows, build_weights, ROOT

lookup = block_lookup()
print("YEAR", YEAR)
print("Blocks in the NJ+NY crosswalk:", len(lookup))


## 2. Build the two commute pieces and stack them

- NJ main: home and work both in New Jersey.
- NJ aux: work in New Jersey, and the home block is in Bronx, Brooklyn, Manhattan, Queens, or Staten Island. The rest of New York State, and every other state, is dropped.
- Same-ZCTA rows stay in the flow file. They are marked `same_zcta`.


In [ ]:
flows, audit = build_flows(lookup)
print(audit.to_string(index=False))
print("Rows including same-ZCTA:", len(flows))
print("Same-ZCTA jobs:", int(flows.loc[flows.same_zcta, "commute_jobs"].sum()))


## 3. Weights

Drop same-ZCTA rows. For each New Jersey workplace:

`inbound_weight = jobs on this home-to-work link / all outside jobs into that workplace`

The weights for one workplace sum to 1.


In [ ]:
weights = build_weights(flows)
out = Path("Dhruvesh/commute_outputs")
flows.to_csv(out / f"nj_inbound_flows_{YEAR}.csv", index=False)
weights.to_csv(out / f"nj_inbound_weights_{YEAR}.csv", index=False)
audit.to_csv(out / f"nj_inbound_audit_{YEAR}.csv", index=False)

print("Outside links:", len(weights))
print("NJ workplaces:", weights.destination_zcta.nunique())
print(weights.groupby("origin_state").agg(links=("origin_zcta", "size"), jobs=("commute_jobs", "sum")))
print("Weight check (min, max):", weights.groupby("destination_zcta").inbound_weight.sum().agg(["min", "max"]).to_list())
weights.head(10)


## 4. Pressure, once you have cases

Put a file at `Dhruvesh/Data/cases/cases.csv` with columns `zipcode`, `date`, `cases`.
Those zip codes are **home** areas (NJ and NYC), not only the workplace.

`pressure = sum of inbound_weight × that home area's cases that day`

If a home area that feeds the workplace has no case row that day, `pressure` is left blank. `pressure_observed` is the sum over the homes that did have a row, and `weight_coverage` is the share of the workplace's outside jobs those homes account for.

```python
from build_nj_inbound_weights import pressure_from_cases
import pandas as pd
cases = pd.read_csv("Dhruvesh/Data/cases/cases.csv")
pressure = pressure_from_cases(weights, cases)
# columns: zipcode, date, pressure, pressure_observed, weight_coverage
```
